# INDPRO ARIMA Analysis

This notebook explains the forecasting workflow used by the production pipeline. The code in `src/` is the source of truth; the notebook is a readable walkthrough for exploration and portfolio review.

## 1. Dataset Overview

The dataset contains monthly U.S. Industrial Production Index observations from FRED. The target column is `INDPRO`, indexed by `observation_date`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data import load_series
from src.diagnostics import adf_test, residual_diagnostics
from src.evaluation import metrics, moving_average_forecast, naive_forecast
from src.modeling import fit_arima, forecast_on_test, ic_grid_search, model_name, select_candidate_orders
from src.preprocessing import descriptive_stats, first_difference, train_test_split_series

series = load_series(PROJECT_ROOT / "INDPRO.csv", "observation_date", "INDPRO")
series.head(), series.tail(), series.shape

## 2. Time Series Plot

The level series shows long-run growth, recessionary declines, and a sharp COVID-period shock.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
series.plot(ax=ax, linewidth=1.8)
ax.set_title("Industrial Production Index (INDPRO)")
ax.set_xlabel("Date")
ax.set_ylabel("INDPRO")
ax.grid(alpha=0.25)
plt.show()

## 3. Stationarity

The ARIMA workflow uses first-order differencing to remove the non-stationary level trend. The Augmented Dickey-Fuller test checks whether the differenced series rejects the unit-root null hypothesis.

In [ ]:
diff = first_difference(series)
display(descriptive_stats(diff))
adf_result = adf_test(diff)
adf_result

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
diff.plot(ax=ax, linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8, alpha=0.5)
ax.set_title("First Difference of INDPRO")
ax.set_xlabel("Date")
ax.set_ylabel("Difference")
ax.grid(alpha=0.25)
plt.show()

## 4. ACF/PACF

ACF and PACF plots guide low-order ARMA candidate selection for the differenced series.

In [ ]:
x = diff.dropna()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_acf(x, lags=24, ax=axes[0])
plot_pacf(x, lags=24, method="ywm", ax=axes[1])
axes[0].set_title("ACF of First Difference")
axes[1].set_title("PACF of First Difference")
plt.tight_layout()
plt.show()

## 5. Model Selection

The series is split chronologically. ARMA candidates are estimated on the differenced training series and ranked with AIC, BIC, and HQIC.

In [ ]:
train, test = train_test_split_series(series, 0.80)
diff_train = first_difference(train).dropna()
ic_table = ic_grid_search(diff_train, max_p=4, max_q=4)
display(ic_table.head(10))
candidates = select_candidate_orders(ic_table, top_k=3)
candidates

## 6. Evaluation

ARIMA candidates are compared against simple baselines: a naive forecast and a trailing moving-average forecast.

In [ ]:
rows = []
baseline_predictions = {
    "Naive forecast": naive_forecast(train, test),
    "Moving average (12) forecast": moving_average_forecast(train, test, window=12),
}
for name, forecast in baseline_predictions.items():
    rows.append({"Model": name, "ModelType": "Baseline", **metrics(test, forecast)})

fitted_models = {}
for label, p, q in candidates:
    name = model_name(p, 1, q, label)
    fit_result = fit_arima(train, (p, 1, q))
    forecast, interval = forecast_on_test(fit_result, test)
    fitted_models[name] = (fit_result, forecast, interval)
    rows.append({"Model": name, "ModelType": "ARIMA", **metrics(test, forecast)})

comparison = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
comparison[["Model", "ModelType", "RMSE", "MAE", "MAPE_percent", "Theil_U2"]]

## 7. Residual Diagnostics

The best ARIMA model is checked for residual autocorrelation and non-Gaussian crisis-period tails.

In [ ]:
best_arima_name = comparison[comparison["ModelType"] == "ARIMA"].iloc[0]["Model"]
best_fit, best_forecast, best_interval = fitted_models[best_arima_name]
residual_diagnostics(best_fit)

In [ ]:
residuals = pd.Series(best_fit.resid).dropna()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
residuals.plot(ax=axes[0], linewidth=1.3)
axes[0].axhline(0, color="black", linewidth=0.8, alpha=0.5)
axes[0].set_title(f"Residuals: {best_arima_name}")
plot_acf(residuals, lags=24, ax=axes[1])
axes[1].set_title("Residual ACF")
plt.tight_layout()
plt.show()

## 8. Forecast Interpretation

The final forecast should be interpreted as a statistical extrapolation from historical dynamics, not as a structural macroeconomic forecast. Large external shocks can move industrial production outside the range implied by a univariate ARIMA model.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
train.plot(ax=ax, label="Train", linewidth=1.4)
test.plot(ax=ax, label="Test", linewidth=1.4)
best_forecast.plot(ax=ax, label="ARIMA forecast", linewidth=1.8)
ax.fill_between(best_interval.index, best_interval.iloc[:, 0], best_interval.iloc[:, 1], alpha=0.2, label="95% interval")
ax.set_title(f"Test Forecast: {best_arima_name}")
ax.set_xlabel("Date")
ax.set_ylabel("INDPRO")
ax.grid(alpha=0.25)
ax.legend()
plt.show()